# ChE629 Course Project

This folder contains more information about the proteins (target). Each file contains a dictionary with three keys - 

1) amino acid sequence of the protein, 
2) matrix containing features about the sequence - size is len_sequence * 67
3) number of nodes - or simply the len_sequence

url = "https://github.com/macjiffriya/EDTA/tree/main/data/kiba/Potential_Residue_Feature"

In [1]:
import torch
import numpy as np
import pandas as pd
import sklearn
import os
import pickle

In [2]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem
    print("RDKit already installed")
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit"])
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem
    print("RDKit installed and imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 52.4 MB/s eta 0:00:00
RDKit installed and imported successfully


In [3]:
try:
    from Bio import SeqIO
    print("Biopython already installed")
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])
    from Bio import SeqIO
    print("Biopython installed and imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.4 MB/s eta 0:00:00
Biopython installed and imported successfully


In [4]:
print("Environment works")

Environment works


In [5]:
# Local paths
davis_path = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/SMILES_affinity.csv"
kiba_path = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/SMILES_affinity.csv"

# Load datasets
davis_data = pd.read_csv(davis_path)
kiba_data = pd.read_csv(kiba_path)

print("DAVIS shape:", davis_data.shape)
print("KIBA shape:", kiba_data.shape)

DAVIS shape: (30056, 3)
KIBA shape: (118254, 3)


Davis dataset 'label' value is already transformed according to the paper. Kd --> pKd

#### Input Features Processing

a) Drug SMILES

This character level enconding could be replaced with atom level encoding. Like convert C,l to Cl and B,r to Br respectively.

In [6]:
davis_data['ligand'].head()

0    CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=C...
1    CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=C...
2    CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=C...
3    CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=C...
4    CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OC[C@H](CC4=C...
Name: ligand, dtype: object

In [7]:
# Combine SMILES from both datasets
combined_smiles = pd.concat([davis_data['ligand'], kiba_data['ligand']])

# Build vocabulary
chars = sorted(set("".join(combined_smiles)))
char_to_int = {c: i+1 for i, c in enumerate(chars)}

print("Vocabulary size:", len(chars))
print("Characters:", chars)

Vocabulary size: 32
Characters: ['#', '(', ')', '+', '-', '.', '/', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'l', 'r']


We can change the max length of the SMILES, though it won't effect too much.

In [8]:
# Encoding function
def encode_smiles(smiles):
    return [char_to_int[c] for c in smiles]

# Set max length
max_len = 100
print("Using max SMILES length:", max_len)

# Pad / truncate
def pad_or_truncate(vec):
    if len(vec) > max_len:
        return vec[:max_len]
    return vec + [0]*(max_len - len(vec))

# ---- DAVIS ----
davis_vectors = davis_data['ligand'].apply(encode_smiles).apply(pad_or_truncate)
davis_smiles_tensor = torch.tensor(np.stack(davis_vectors.values), dtype=torch.long)
print("Davis tensor shape:", davis_smiles_tensor.shape)

# ---- KIBA ----
kiba_vectors = kiba_data['ligand'].apply(encode_smiles).apply(pad_or_truncate)
kiba_smiles_tensor = torch.tensor(np.stack(kiba_vectors.values), dtype=torch.long)
print("KIBA tensor shape:", kiba_smiles_tensor.shape)

Using max SMILES length: 100
Davis tensor shape: torch.Size([30056, 100])
KIBA tensor shape: torch.Size([118254, 100])


b) Drug Features

In [9]:
max_atoms = 50
feature_dim = 24

print(f'Maximum atoms for features are taken into account (per molecule): {max_atoms}')

def pad_or_truncate(mat):
    num_atoms = mat.shape[0]

    # truncate
    if num_atoms > max_atoms:
        return mat[:max_atoms, :]

    # pad
    if num_atoms < max_atoms:
        pad_rows = max_atoms - num_atoms
        padding = np.zeros((pad_rows, feature_dim))
        return np.vstack([mat, padding])

    return mat


def load_drug_tensor(pkl_path):

    daf = pd.read_pickle(pkl_path)

    # pad/truncate all molecules
    daf_fixed = {k: pad_or_truncate(v) for k, v in daf.items()}

    # stack matrices
    matrix_stack = np.stack(list(daf_fixed.values()))

    # convert to tensor
    drug_tensor = torch.tensor(matrix_stack, dtype=torch.float32)

    return drug_tensor, list(daf_fixed.keys())


# ===============================
# KIBA
# ===============================

kiba_path = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/Drug_Atomic_Feature24.pkl"

kiba_drug_tensor, kiba_drug_keys = load_drug_tensor(kiba_path)

print("KIBA tensor shape:", kiba_drug_tensor.shape)


# ===============================
# DAVIS
# ===============================

davis_path = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/Drug_Atomic_Feature24.pkl"

davis_drug_tensor, davis_drug_keys = load_drug_tensor(davis_path)

print("DAVIS tensor shape:", davis_drug_tensor.shape)

Maximum atoms for features are taken into account (per molecule): 50
KIBA tensor shape: torch.Size([2111, 50, 24])
DAVIS tensor shape: torch.Size([68, 50, 24])


c) Target Feature + BPE

In [10]:
# =====================================================
# DATASET PATHS
# =====================================================

DAVIS_PATH = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/Target_Feature+BPE"
KIBA_PATH  = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/Target_Feature+BPE"

# =====================================================
# CONSTANTS (from EDTA paper)
# =====================================================

MAX_TARGET_LEN = 500
MAX_BPE_LEN_DAVIS = 400
MAX_BPE_LEN_KIBA = 556


# =====================================================
# PADDING FUNCTIONS
# =====================================================

def pad_target_feature(x):
    
    x = np.asarray(x)
    L, d = x.shape
    
    if L >= MAX_TARGET_LEN:
        return x[:MAX_TARGET_LEN]

    padded = np.zeros((MAX_TARGET_LEN, d))
    padded[:L] = x
    return padded


def pad_bpe(x, max_len):

    x = np.asarray(x)
    L, d = x.shape

    if L >= max_len:
        return x[:max_len]

    padded = np.zeros((max_len, d))
    padded[:L] = x
    return padded


# =====================================================
# LOAD TARGET FEATURES
# =====================================================

def load_target_features(folder_path, bpe_len):

    target_features = []
    target_bpe = []
    protein_ids = []

    files = sorted(os.listdir(folder_path))

    for file in files:

        if not file.endswith(".pkl"):
            continue

        protein_id = file.replace(".pkl", "")
        protein_ids.append(protein_id)

        file_path = os.path.join(folder_path, file)

        with open(file_path, "rb") as f:
            data = pickle.load(f)

        seq_feat = pad_target_feature(data["seq_feat"])
        bpe_feat = pad_bpe(data["token_representation"], bpe_len)

        target_features.append(seq_feat)
        target_bpe.append(bpe_feat)

    target_features = torch.tensor(np.stack(target_features), dtype=torch.float32)
    target_bpe = torch.tensor(np.stack(target_bpe), dtype=torch.float32)

    return target_features, target_bpe, protein_ids


# =====================================================
# LOAD DAVIS
# =====================================================

print("Loading DAVIS proteins...")

davis_feat, davis_bpe, davis_ids = load_target_features(
    DAVIS_PATH,
    MAX_BPE_LEN_DAVIS
)

print("DAVIS loaded")
print("Proteins:", len(davis_ids))
print("Target feature tensor:", davis_feat.shape)
print("BPE tensor:", davis_bpe.shape)


# =====================================================
# LOAD KIBA
# =====================================================

print("\nLoading KIBA proteins...")

kiba_feat, kiba_bpe, kiba_ids = load_target_features(
    KIBA_PATH,
    MAX_BPE_LEN_KIBA
)

print("KIBA loaded")
print("Proteins:", len(kiba_ids))
print("Target feature tensor:", kiba_feat.shape)
print("BPE tensor:", kiba_bpe.shape)


# =====================================================
# SANITY CHECK
# =====================================================

print("\nExample DAVIS protein:", davis_ids[0])
print("Feature shape:", davis_feat[0].shape)
print("BPE shape:", davis_bpe[0].shape)

print("\nExample KIBA protein:", kiba_ids[0])
print("Feature shape:", kiba_feat[0].shape)
print("BPE shape:", kiba_bpe[0].shape)

Loading DAVIS proteins...
DAVIS loaded
Proteins: 442
Target feature tensor: torch.Size([442, 500, 42])
BPE tensor: torch.Size([442, 400, 1280])

Loading KIBA proteins...
KIBA loaded
Proteins: 228
Target feature tensor: torch.Size([228, 500, 42])
BPE tensor: torch.Size([228, 556, 1280])

Example DAVIS protein: AAK1
Feature shape: torch.Size([500, 42])
BPE shape: torch.Size([400, 1280])

Example KIBA protein: O00141
Feature shape: torch.Size([500, 42])
BPE shape: torch.Size([556, 1280])


d) Potential Residue

In [11]:
# =====================================================
# PATHS
# =====================================================
davis_residue_path = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/Potential_Residue_Feature"
kiba_residue_path  = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/Potential_Residue_Feature"

# =====================================================
# CONSTANTS
# =====================================================
MAX_RESIDUE_LEN = 200
FEATURE_DIM = 67

# =====================================================
# PADDING FUNCTION
# =====================================================

def pad_residue_feature(mat):
    mat = np.asarray(mat)
    L, d = mat.shape

    # truncate
    if L >= MAX_RESIDUE_LEN:
        return mat[:MAX_RESIDUE_LEN]

    # pad
    padded = np.zeros((MAX_RESIDUE_LEN, d))
    padded[:L] = mat
    return padded

# =====================================================
# LOADER FUNCTION
# =====================================================

def load_residue_tensor(folder_path):
    residue_features = []
    protein_ids = []

    files = sorted(os.listdir(folder_path))
    for file in files:
        if not file.endswith(".pkl"):
            continue

        protein_id = file.replace(".pkl", "")
        protein_ids.append(protein_id)
        path = os.path.join(folder_path, file)
        with open(path, "rb") as f:
            data = pickle.load(f)

        residue_mat = data["seq_feat"]      # (L, 67)
        residue_mat = pad_residue_feature(residue_mat)
        residue_features.append(residue_mat)

    residue_tensor = torch.tensor(
        np.stack(residue_features),
        dtype=torch.float32
    )
    return residue_tensor, protein_ids

# =====================================================
# LOAD DAVIS
# =====================================================

print("Loading DAVIS residue features...")
davis_residue_tensor, davis_residue_ids = load_residue_tensor(davis_residue_path)
print("DAVIS residue tensor shape:", davis_residue_tensor.shape)

# =====================================================
# LOAD KIBA
# =====================================================

print("\nLoading KIBA residue features...")
kiba_residue_tensor, kiba_residue_ids = load_residue_tensor(kiba_residue_path)
print("KIBA residue tensor shape:", kiba_residue_tensor.shape)

Loading DAVIS residue features...
DAVIS residue tensor shape: torch.Size([442, 200, 67])

Loading KIBA residue features...
KIBA residue tensor shape: torch.Size([229, 200, 67])


Handling the missing Target_feature+BPE protein in KIBA

In [12]:
# ============================================
# FIX KIBA PROTEIN ALIGNMENT — ZERO PLACEHOLDER
# ============================================

kiba_bpe_set     = set(kiba_ids)
kiba_residue_set = set(kiba_residue_ids)
missing_from_bpe = kiba_residue_set - kiba_bpe_set
print(f"Proteins missing from BPE/Target_Feature: {missing_from_bpe}")
assert 'P78527' in missing_from_bpe

# Find where P78527 sits in the residue ID list
# (this determines the row position it should occupy in the feature tensors)
p78527_residue_row = kiba_residue_ids.index('P78527')
print(f"P78527 position in residue ID list: {p78527_residue_row}")

# ---- Build zero placeholder rows ----
# target feat : (1, MAX_TARGET_LEN, feat_dim)
# target bpe  : (1, MAX_BPE_LEN_KIBA, bpe_dim)
zero_target_feat = torch.zeros(
    1, kiba_feat.shape[1], kiba_feat.shape[2],
    dtype=kiba_feat.dtype
)
zero_target_bpe = torch.zeros(
    1, kiba_bpe.shape[1], kiba_bpe.shape[2],
    dtype=kiba_bpe.dtype
)

# ---- Insert placeholder at the correct position ----
# so that kiba_ids and kiba_residue_ids become aligned
kiba_feat = torch.cat([
    kiba_feat[:p78527_residue_row],
    zero_target_feat,
    kiba_feat[p78527_residue_row:]
], dim=0)

kiba_bpe = torch.cat([
    kiba_bpe[:p78527_residue_row],
    zero_target_bpe,
    kiba_bpe[p78527_residue_row:]
], dim=0)

# Insert P78527 into kiba_ids at the matching position
kiba_ids.insert(p78527_residue_row, 'P78527')

# ---- Verify alignment ----
assert kiba_ids == kiba_residue_ids, \
    "KIBA protein ID lists still misaligned after fix"
assert kiba_feat.shape[0] == kiba_bpe.shape[0] == kiba_residue_tensor.shape[0] == len(kiba_ids), \
    "KIBA tensor row counts misaligned after fix"

print(f"KIBA proteins after fix : {len(kiba_ids)}")
print(f"kiba_feat shape         : {kiba_feat.shape}")
print(f"kiba_bpe shape          : {kiba_bpe.shape}")
print(f"kiba_residue shape      : {kiba_residue_tensor.shape}")
print(f"P78527 row in tensors   : {p78527_residue_row}  (zero-padded)")

# ---- Davis alignment check ----
assert davis_ids == davis_residue_ids, \
    f"DAVIS protein ID lists misaligned: {set(davis_ids) ^ set(davis_residue_ids)}"
print("DAVIS protein alignment : OK")

Proteins missing from BPE/Target_Feature: {'P78527'}
P78527 position in residue ID list: 128
KIBA proteins after fix : 229
kiba_feat shape         : torch.Size([229, 500, 42])
kiba_bpe shape          : torch.Size([229, 556, 1280])
kiba_residue shape      : torch.Size([229, 200, 67])
P78527 row in tensors   : 128  (zero-padded)
DAVIS protein alignment : OK


#### Pair expansion of collected features

In [13]:
# ============================================
# CREATE INDEX MAPS
# ============================================
def make_index_map(keys):
    return {k: i for i, k in enumerate(keys)}

davis_drug_index   = make_index_map(davis_drug_keys)
davis_target_index = make_index_map(davis_ids)
kiba_drug_index    = make_index_map(kiba_drug_keys)
kiba_target_index  = make_index_map(kiba_ids)

#### PyTorch dataset construction

In [14]:
import ast
import torch
from torch.utils.data import Dataset, DataLoader, Subset

# ============================================
# LOAD OFFICIAL SPLIT FILES
# ============================================
def load_fold_indices(path):
    with open(path, "r") as f:
        content = f.read().strip()
    return ast.literal_eval(content)

# test_fold_setting1.txt  → flat list of indices
# train_fold_setting1.txt → list of 5 lists (one per fold)
davis_test_indices  = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/folds/test_fold_setting1.txt")
davis_train_folds   = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/folds/train_fold_setting1.txt")

kiba_test_indices   = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/folds/test_fold_setting1.txt")
kiba_train_folds    = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/folds/train_fold_setting1.txt")

print(f"DAVIS test set size      : {len(davis_test_indices)}")
print(f"DAVIS folds (5)          : {[len(f) for f in davis_train_folds]}")
print(f"KIBA  test set size      : {len(kiba_test_indices)}")
print(f"KIBA  folds (5)          : {[len(f) for f in kiba_train_folds]}")

DAVIS test set size      : 5010
DAVIS folds (5)          : [5010, 5009, 5009, 5009, 5009]
KIBA  test set size      : 19709
KIBA  folds (5)          : [19709, 19709, 19709, 19709, 19709]


In [15]:
# ============================================
# PYTORCH DATASET  (unchanged from before)
# ============================================
class DTIDataset(Dataset):
    """
    Stores only index arrays. Slices base feature tensors on demand
    in __getitem__ — no duplication of feature matrices in memory.
    """
    def __init__(
        self,
        d_indices, t_indices, labels,
        smiles_tensor,
        drug_atomic_tensor,
        target_feat_tensor,
        target_bpe_tensor,
        residue_tensor
    ):
        assert len(d_indices) == len(t_indices) == len(labels)
        self.d_indices          = d_indices
        self.t_indices          = t_indices
        self.labels             = labels
        self.smiles_tensor      = smiles_tensor
        self.drug_atomic_tensor = drug_atomic_tensor
        self.target_feat_tensor = target_feat_tensor
        self.target_bpe_tensor  = target_bpe_tensor
        self.residue_tensor     = residue_tensor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        d_i = self.d_indices[idx]
        t_i = self.t_indices[idx]
        return {
            "smiles"      : self.smiles_tensor[d_i],
            "drug_atomic" : self.drug_atomic_tensor[d_i],
            "target_feat" : self.target_feat_tensor[t_i],
            "target_bpe"  : self.target_bpe_tensor[t_i],
            "residue"     : self.residue_tensor[t_i],
            "label"       : self.labels[idx]
        }

# ============================================
# BUILD FULL DATASETS (pair indices + labels)
# ============================================
def build_pair_indices(df, drug_map, target_map):
    df = df.copy()
    df["d_idx"] = df["ligand"].map(drug_map)
    df["t_idx"] = df["protein"].map(target_map)
    df = df.dropna(subset=["d_idx", "t_idx"])
    df[["d_idx", "t_idx"]] = df[["d_idx", "t_idx"]].astype(int)
    d_indices = torch.from_numpy(df["d_idx"].values.astype("int64"))
    t_indices = torch.from_numpy(df["t_idx"].values.astype("int64"))
    labels    = torch.tensor(df["label"].values, dtype=torch.float32)
    return d_indices, t_indices, labels

davis_d_idx, davis_t_idx, davis_labels = build_pair_indices(davis_data, davis_drug_index, davis_target_index)
kiba_d_idx,  kiba_t_idx,  kiba_labels  = build_pair_indices(kiba_data,  kiba_drug_index,  kiba_target_index)

davis_dataset = DTIDataset(
    davis_d_idx, davis_t_idx, davis_labels,
    davis_smiles_tensor, davis_drug_tensor,
    davis_feat, davis_bpe, davis_residue_tensor
)

kiba_dataset = DTIDataset(
    kiba_d_idx, kiba_t_idx, kiba_labels,
    kiba_smiles_tensor, kiba_drug_tensor,
    kiba_feat, kiba_bpe, kiba_residue_tensor
)

print(f"\nDAVIS full dataset size : {len(davis_dataset)}")
print(f"KIBA  full dataset size : {len(kiba_dataset)}")


DAVIS full dataset size : 30056
KIBA  full dataset size : 118254


#### Sample level setting

In [16]:
# ============================================
# FIXED SPLIT UTILITY USING OFFICIAL INDICES
# ============================================
def make_official_splits(dataset, train_folds, test_indices):
    """
    Uses the official fold index files to produce:
      - test_set     : Subset using the fixed test indices
      - fold_splits  : List of 5 (train_subset, val_subset) tuples
                       following the 4-train / 1-val rotation
    All subsets reference the same base dataset — no data is copied.
    """
    test_set = Subset(dataset, test_indices)

    fold_splits = []
    for val_fold_idx in range(5):
        val_indices   = train_folds[val_fold_idx]
        train_indices = []
        for i, fold in enumerate(train_folds):
            if i != val_fold_idx:
                train_indices.extend(fold)

        train_subset = Subset(dataset, train_indices)
        val_subset   = Subset(dataset, val_indices)
        fold_splits.append((train_subset, val_subset))

    return test_set, fold_splits

davis_test_set, davis_fold_splits = make_official_splits(davis_dataset, davis_train_folds, davis_test_indices)
kiba_test_set,  kiba_fold_splits  = make_official_splits(kiba_dataset,  kiba_train_folds,  kiba_test_indices)

# Print split sizes for all folds
print("\nDAVIS splits:")
print(f"  Test : {len(davis_test_set)}")
for i, (tr, vl) in enumerate(davis_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")

print("\nKIBA splits:")
print(f"  Test : {len(kiba_test_set)}")
for i, (tr, vl) in enumerate(kiba_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")


DAVIS splits:
  Test : 5010
  Fold 1 → train=20036  val=5010
  Fold 2 → train=20037  val=5009
  Fold 3 → train=20037  val=5009
  Fold 4 → train=20037  val=5009
  Fold 5 → train=20037  val=5009

KIBA splits:
  Test : 19709
  Fold 1 → train=78836  val=19709
  Fold 2 → train=78836  val=19709
  Fold 3 → train=78836  val=19709
  Fold 4 → train=78836  val=19709
  Fold 5 → train=78836  val=19709


Dataloaders

In [17]:
# ============================================
# DATALOADER FACTORY
# ============================================
BATCH_SIZE  = 32
NUM_WORKERS = 2

def make_loaders(train_subset, val_subset, test_subset):
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_subset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

# Build loaders for all 5 folds — ready for cross-validation training loop
davis_loaders = [make_loaders(tr, vl, davis_test_set) for tr, vl in davis_fold_splits]
kiba_loaders  = [make_loaders(tr, vl, kiba_test_set)  for tr, vl in kiba_fold_splits]

print("\nDataLoaders ready for 5-fold cross-validation")

# ============================================
# SANITY CHECK — one batch from fold 0
# ============================================
def check_batch(loader, name):
    batch = next(iter(loader))
    print(f"\n{name} sample batch:")
    for k, v in batch.items():
        print(f"  {k:15s} : {v.shape}  dtype={v.dtype}")

davis_train_loader_0, davis_val_loader_0, davis_test_loader = davis_loaders[0]
kiba_train_loader_0,  kiba_val_loader_0,  kiba_test_loader  = kiba_loaders[0]

check_batch(davis_train_loader_0, "DAVIS fold-0 train")
check_batch(davis_val_loader_0,   "DAVIS fold-0 val")
check_batch(davis_test_loader,    "DAVIS test")
check_batch(kiba_train_loader_0,  "KIBA  fold-0 train")


DataLoaders ready for 5-fold cross-validation


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



DAVIS fold-0 train sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400, 1280])  dtype=torch.float32
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS fold-0 val sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400, 1280])  dtype=torch.float32
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS test sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  tar